## Imports
Standard libraries plus the OpenAI SDK, Gradio for the UI, SQLite for storing ticket prices, and Pillow for handling generated images.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import base64 
from io import BytesIO
from PIL import Image 

## Load API Key & Configure Model
Loads the OpenAI API key from `.env` and sets the chat model to use (`gpt-4.1-mini`).

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


## Set Up Ticket Price Database
Creates a local SQLite database (`prices.db`) if it doesn't already exist, and seeds it with sample ticket prices for a few cities.

In [3]:
DB = "prices.db"

with sqlite3.connect(DB) as conn :
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (? , ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

## System Prompt
Defines FlightAI's persona and response style — short, courteous, and accurate answers.

In [4]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

## Ticket Price Lookup Tool
A Python function that queries the SQLite database for a city's ticket price. This is the function the model will call via tool use.

In [5]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

## Tool Definition
Describes `get_ticket_price` to the OpenAI API in the JSON schema format it expects, so the model knows when and how to call it.

In [6]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

## Image Generation
Generates a vibrant pop-art style illustration of a destination city using OpenAI's image model.

In [7]:
def artist(city):
    image_response = openai.images.generate(
        model = "gpt-image-1-mini",
        prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        size = "1024x1024",
        n = 1,
      )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

## Text-to-Speech
Converts the assistant's text reply into spoken audio using OpenAI's TTS model.

In [8]:
def talker(message):
    response = openai.audio.speech.create(
        model = "gpt-4o-mini-tts",
        voice = "onyx",
        input = message 
    )
    return response.content 

## Tool Call Handler
Processes all tool calls returned by the model in a single turn, fetches the relevant data (ticket prices), and tracks which cities were mentioned so an image can be generated later.

In [9]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id" : tool_call.id
            })
    return responses, cities 

## Core Chat Logic
Sends the conversation to the model, handles any tool calls in a loop until the model produces a final answer, then generates matching audio and (if relevant) an image.

In [10]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history 
    response = openai.chat.completions.create(model = MODEL, messages = messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message 
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message.model_dump())
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])

    return history, voice, image 

## Gradio UI
Builds the multi-output interface: a chatbot window, an image panel, an audio player, and a message box — wired together with Gradio event handlers.

In [11]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]


with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height = 500, type = "messages")
        image_output = gr.Image(height=500, interactive= False)
    with gr.Row():  
        audio_output = gr.Audio(autoplay= True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs= [message, chatbot]).then(
        chat, inputs= chatbot, outputs = [chatbot, audio_output, image_output]
    )

## Launch App
Starts the Gradio app in the browser.

In [12]:
ui.launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
